In [48]:
import numpy as np

class Tensor:

    def __init__(self, data, children=()):
        self.data = data
        self.children = set(children)
        self.grad = np.zeros(data.shape)
        self.backwards = lambda : None

    def __add__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(np.array[other])
        out = Tensor(self.data + other.data, (self, other))

        def backwards():
            self.grad += unbroadcast(out.grad, self.data.shape)
            other.grad += unbroadcast(out.grad, other.data.shape)

        out.backwards = backwards
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(np.array[other])
        out = Tensor(self.data * other.data, (self, other))

        def backwards():
            self.grad += unbroadcast(out.grad * other.data, self.data.shape)
            other.grad += unbroadcast(out.grad * self.data, other.data.shape)

        out.backwards = backwards
        return out

    def __matmul__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(np.array[other])
        out = Tensor(self.data @ other.data, (self, other))

        def backwards():
            self.grad += out.grad @ other.data.T
            other.grad += self.data.T @ out.grad

        out.backwards = backwards
        return out

    def tanh(self):
        x = self.data
        t = (np.exp(2*x) - 1) / (np.exp(2*x) + 1)
        out = Tensor(t, (self, ))

        def backwards():
            self.grad += (1 - t**2) * out.grad

        out.backwards = backwards
        return out

    def __repr__(self):
        return f'Data:\n{self.data}\nGrad:\n{self.grad}'

    def sum(self, axis=0):
        out = Tensor(self.data.sum(axis=axis), (self, ))

        def backwards():
            self.grad += np.ones_like(self.data) * np.expand_dims(out.grad, axis=axis)

        out.backwards = backwards
        return out

    def backward(self):
        topo = []
        visited = set()

        def topo_sort(node):
            if node not in visited:
                visited.add(node)
                for child in node.children:
                    topo_sort(child)
                topo.append(node)

        self.grad = np.ones(self.data.shape)
        topo_sort(self)

        for node in reversed(topo):
            node.backwards()
        

def unbroadcast(grad, target_shape):
    while grad.ndim > len(target_shape):
        grad = grad.sum(axis=0)
    for axis, size in enumerate(target_shape):
        if size == 1:
            grad = grad.sum(axis=axis, keepdims=True)
    return grad
        

class Layer:

    def __init__(self, nin, nout):
        self.weights = Tensor(np.random.uniform(-1, 1, (nin, nout)))
        self.bias = Tensor(np.random.uniform(-1, 1, nout))

    def __call__(self, x):
        act = x @ self.weights + self.bias
        return act.tanh()

class MLP:

    def __init__(self, nin, nouts):
        size = [nin] + nouts
        self.layers = [Layer(size[i], size[i + 1]) for i in range(len(nouts))]
        self.parameters = []
        for layer in self.layers:
            self.parameters.append(layer.weights)
            self.parameters.append(layer.bias)

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

x = Tensor(np.array([[0.4, 0.2, 0.5, 0.8],
                     [0.6, 0.5, 0.6, 0.8],
                     [0.8, 0.2, 0.1, 0.8]]))

model = MLP(4, [5, 6, 1])

loss = model(x)

loss.backward()

model.parameters


[Data:
 [[ 0.67072084  0.84045763 -0.48475634  0.26250832 -0.59783133]
  [-0.90964568 -0.7631677  -0.5524107  -0.34026033  0.94061397]
  [ 0.31652475 -0.67946475 -0.4118926  -0.14925504  0.82969163]
  [-0.83270224 -0.49560059 -0.69230826  0.69273864  0.61281543]]
 Grad:
 [[-0.08153684  0.3144753   0.02655179  0.04206397  0.02158043]
  [-0.04243822  0.19195841  0.01358908  0.02591594  0.00908893]
  [-0.05858821  0.27802203  0.02031396  0.03689336  0.01139502]
  [-0.1116421   0.45775876  0.03803048  0.06061874  0.02729887]],
 Data:
 [-0.77942354  0.63207985 -0.78528848 -0.21193028  0.70268226]
 Grad:
 [-0.13955262  0.57219845  0.0475381   0.07577342  0.03412359],
 Data:
 [[-0.15397343 -0.08755837 -0.83750211 -0.18267134 -0.9444988  -0.92536517]
  [-0.33251386  0.77941534 -0.78198719  0.29326953  0.70731774  0.22342731]
  [ 0.62751085  0.81146245 -0.95190113  0.38026424  0.41783051  0.05717516]
  [ 0.56466217 -0.13501804 -0.17354482 -0.41766682  0.61020628  0.4817443 ]
  [-0.9921498   0.2